# Chapter 13 — GPTQ Quantization

A 70B parameter model in FP16 requires 140GB of VRAM. Quantization compresses weights to INT8 or INT4, enabling LLMs on consumer hardware. This chapter covers post-training quantization theory and practice.

### Glossary / Glossário

• quantization — reducing numerical precision of model weights to save memory and speed up inference. / Redução da precisão numérica dos pesos para economizar memória e acelerar inferência.
• INT8 — 8-bit integer representation for quantized weights. / Representação inteira de 8 bits para pesos quantizados.
• INT4 — 4-bit integer representation for aggressive quantization. / Representação inteira de 4 bits para quantização agressiva.
• GPTQ — post-training quantization method using approximate second-order information. / Método de quantização pós-treinamento usando informação de segunda ordem aproximada.
• AWQ — Activation-aware Weight Quantization, preserves salient weights during quantization. / Quantização de pesos ciente de ativações, preserva pesos importantes durante quantização.
• GGUF — file format for quantized models used by llama.cpp. / Formato de arquivo para modelos quantizados usado pelo llama.cpp.
• PTQ — Post-Training Quantization, quantizing a model after training is complete. / Quantização pós-treinamento, quantizar modelo após treinamento completo.
• QAT — Quantization-Aware Training, training with simulated quantization for better accuracy. / Treinamento ciente de quantização, treinar com quantização simulada para melhor acurácia.

In [ ]:
import torch

print("=== Weight Quantization: FP32 -> INT4 ===\n")

torch.manual_seed(42)
W = torch.randn(256, 256) * 0.02  # Original weight matrix - model parameters to compress

bits = 4          # Target precision - 4 bits = 16 discrete levels
group_size = 128  # Number of weights quantized together with same scale
qmax = 2**bits - 1  # Maximum quantizable value = 2^bits - 1 = 15
W_deq = torch.zeros_like(W)  # Dequantized weights - reconstructed after quantize/dequantize roundtrip

for g in range(W.shape[1] // group_size):
    group = W[:, g*group_size:(g+1)*group_size]
    gmin, gmax = group.min(), group.max()
    scale = (gmax - gmin) / qmax       # Scaling factor to map float range to integer range
    zero_point = (-gmin / scale).round()  # Offset to handle negative values in unsigned integers
    q = ((group / scale) + zero_point).round().clamp(0, qmax)
    W_deq[:, g*group_size:(g+1)*group_size] = (q - zero_point) * scale

mse = ((W - W_deq) ** 2).mean().item()  # Mean squared error between original and quantized weights
max_err = (W - W_deq).abs().max().item()

print(f"Original:  shape={tuple(W.shape)}, range=[{W.min():.4f}, {W.max():.4f}]")
print(f"INT4 (group_size={group_size}): MSE={mse:.8f}, max_err={max_err:.6f}")

fp32_bytes = W.nelement() * 4
int4_bytes = W.nelement() * 0.5
print(f"\nFP32: {fp32_bytes / 1024:.1f} KB -> INT4: {int4_bytes / 1024:.1f} KB ({fp32_bytes / int4_bytes:.0f}x compression)")

print(f"\nSample (row 0, first 6 weights):")
print(f"  Original:  {[round(v, 4) for v in W[0,:6].tolist()]}")
print(f"  Quantized: {[round(v, 4) for v in W_deq[0,:6].tolist()]}")

# --- Aha demo: Compare different bit widths ---
print("\n=== Aha! Bit Width vs. Quality Trade-off ===\n")
print(f"{'Bits':>4s} | {'Levels':>6s} | {'MSE':>12s} | {'Compression':>11s} | Quality")
print("-" * 65)

for bits_test in [8, 4, 2]:
    qmax_test = 2**bits_test - 1  # Discrete levels for this bit width
    W_test = torch.zeros_like(W)
    for g in range(W.shape[1] // group_size):
        group = W[:, g*group_size:(g+1)*group_size]
        gmin, gmax = group.min(), group.max()
        scale_test = (gmax - gmin) / qmax_test
        zp_test = (-gmin / scale_test).round()
        q_test = ((group / scale_test) + zp_test).round().clamp(0, qmax_test)
        W_test[:, g*group_size:(g+1)*group_size] = (q_test - zp_test) * scale_test
    mse_test = ((W - W_test) ** 2).mean().item()
    bytes_test = W.nelement() * (bits_test / 8)
    compression = fp32_bytes / bytes_test
    quality = "excellent" if mse_test < 1e-7 else "good" if mse_test < 1e-5 else "lossy"
    print(f"{bits_test:>4d} | {qmax_test+1:>6d} | {mse_test:>12.8f} | {compression:>10.1f}x | {quality}")

print("\n-> Fewer bits = more compression but higher error. 4-bit is the sweet spot for LLMs.")

In [ ]:
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig

config = BaseQuantizeConfig(bits=4, group_size=128, desc_act=True)
model = AutoGPTQForCausalLM.from_pretrained("llama-7b", config)
model.quantize(calibration_data)
model.save_quantized("llama-7b-int4")
# 14GB -> 3.5GB, ~97% quality retained

---

**Course:** Aprenda LLM | **Chapter 13**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.